# logging изнутри

Лекция показала, как логировать. Здесь — из чего логи сделаны: что такое
`LogRecord`, кто его правит и кто превращает в текст.

Нужно для домашней задачи `json_logging` — там пишутся свой форматтер и
свой фильтр.

| # | О чём |
|---|---|
| 1 | `LogRecord` — что это |
| 2 | `msg`, `args` и почему не f-строка |
| 3 | `extra` — что можно, что нельзя |
| 4 | `Filter` — правит запись |
| 5 | `Formatter` — свой JSON |
| 6 | `exception` против `error(exc)` |
| 7 | Перехват записей в тестах |
| 8 | `contextvars` — контекст без передачи |

In [ ]:
import json
import logging

# Свой handler, который просто копит записи — так удобно их разглядывать.
class Collector(logging.Handler):
    def __init__(self) -> None:
        super().__init__()
        self.records: list[logging.LogRecord] = []

    def emit(self, record: logging.LogRecord) -> None:
        self.records.append(record)


def fresh_logger(name: str) -> tuple[logging.Logger, Collector]:
    logger = logging.getLogger(name)
    logger.handlers.clear()
    logger.setLevel(logging.DEBUG)
    logger.propagate = False
    collector = Collector()
    logger.addHandler(collector)
    return logger, collector


print("готово")

## 1. `LogRecord` — что это

Каждый вызов `logger.info(...)` создаёт объект `LogRecord`. Он идёт через
фильтры и хендлеры, а форматтер в конце превращает его в строку.

In [ ]:
logger, collector = fresh_logger("demo.record")
logger.info("привет %s", "мир")

record = collector.records[-1]
print(type(record).__name__, "—", len(record.__dict__), "атрибута\n")
for name in sorted(record.__dict__):
    print(f"  {name:18} {record.__dict__[name]!r:.60}")

Здесь всё: уровень, имя логгера, файл и строка, время, процесс, поток.
Форматтер по умолчанию берёт из этого пару полей, остальное выбрасывает.

Для `json_logging` важно обратное: **служебных атрибутов в выводе быть не
должно**. `pathname`, `lineno`, `thread` в каждой строке лога — это
двадцать лишних полей, которые никто не читает.

## 2. `msg`, `args` и почему не f-строка

Шаблон и аргументы хранятся **раздельно**. Подстановка происходит только
когда кто-то позовёт `getMessage()`.

In [ ]:
print("msg          :", record.msg)
print("args         :", record.args)
print("getMessage() :", record.getMessage())

Отсюда правило из лекции: `logger.info("привет %s", name)`, а не
`logger.info(f"привет {name}")`. Причины две.

Первая — цена. Если уровень `INFO` выключен, подстановка не выполнится
вообще. С f-строкой она уже произошла, до вызова.

Вторая важнее для нас: при f-строке шаблон и данные слипаются в одну
строку, и разобрать их обратно невозможно. Сборщик логов не сможет
сгруппировать «одинаковые» записи с разными аргументами.

## 3. `extra` — что можно, что нельзя

`extra={...}` дописывает в запись свои поля.

In [ ]:
logger, collector = fresh_logger("demo.extra")
logger.info("платёж прошёл", extra={"user_id": 42, "amount": 100.5})

record = collector.records[-1]
print("user_id:", record.user_id, "| amount:", record.amount)

Но не любым именем: часть ключей `logging` защищает сам.

In [ ]:
logger, _ = fresh_logger("demo.reserved")

for key in ("levelname", "message", "msg", "name", "asctime", "level", "time", "user_id"):
    try:
        logger.info("x", extra={key: "ЗАХВАЧЕНО"})
        print(f"  extra={{{key!r}: ...}}  -> прошло")
    except KeyError as exc:
        print(f"  extra={{{key!r}: ...}}  -> KeyError: {exc}")

Видно, где граница: `logging` защищает имена **своих атрибутов**
(`levelname`, `msg`, `name`, ...). А `level` и `time` — не его атрибуты,
они проходят свободно.

И вот тут ловушка для `json_logging`: в JSON-выводе ключи как раз
`level` и `time`. Значит **защищать их обязан форматтер**, сам `logging`
этого не сделает:

```python
extra={"level": "DEBUG"}   # запись останется INFO, но в JSON полезет DEBUG
```

если форматтер сложит `extra` поверх своих полей, а не под них.

## 4. `Filter` — правит запись

Фильтр вызывается до хендлеров. Вернул ложь — запись выброшена, вернул
истину — пошла дальше. Но править запись он тоже может, и чаще берут
именно для этого: погасить секрет, добавить поле.

In [ ]:
MASK = "***"


class RedactingFilter(logging.Filter):
    """Гасит секреты в сообщении. Запись не отбрасывает."""

    def __init__(self, secrets: list[str]) -> None:
        super().__init__()
        self.secrets = secrets

    def filter(self, record: logging.LogRecord) -> bool:
        message = record.getMessage()          # уже с подставленными args
        for secret in self.secrets:
            message = message.replace(secret, MASK)

        record.msg = message
        record.args = ()                       # ВАЖНО, см. ниже
        return True


logger, collector = fresh_logger("demo.filter")
logger.addFilter(RedactingFilter(["s3cret-token"]))

logger.info("запрос с token=%s", "s3cret-token")
print(collector.records[-1].getMessage())

Два места, где спотыкаются.

**Первое.** Секрет может прийти не в шаблоне, а в аргументе:
`log.info("token=%s", token)`. В `record.msg` его нет. Поэтому смотреть
надо на `getMessage()`, а не на `record.msg`.

**Второе.** Подменив `msg` готовой строкой, надо обнулить `args` — иначе
следующий `getMessage()` попробует подставить аргументы второй раз:

In [ ]:
broken = logging.LogRecord(
    "x", logging.INFO, "f.py", 1, "token=%s", ("s3cret",), None
)
broken.msg = broken.getMessage().replace("s3cret", MASK)   # args НЕ обнулили

try:
    print(broken.getMessage())
except TypeError as exc:
    print("TypeError:", exc)

## 5. `Formatter` — свой JSON

Форматтер превращает `LogRecord` в строку. Одна запись — одна строка
валидного JSON: собирать её `%`-подстановкой нельзя, первая же кавычка
или перенос строки порвут вывод.

In [ ]:
import datetime as dt

# Атрибуты, которые LogRecord заводит сам. Всё, чего здесь нет, пришло
# через extra — это и надо добавить в вывод.
NATIVE = set(
    logging.LogRecord("", 0, "", 0, "", None, None).__dict__
) | {"message", "asctime", "taskName"}


class JsonFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        payload = {
            "time": dt.datetime.fromtimestamp(record.created, dt.UTC).isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
        }
        if record.exc_info:
            payload["traceback"] = self.formatException(record.exc_info)

        # extra кладём ПОД свои ключи: подменить level или time нельзя.
        for key, value in record.__dict__.items():
            if key not in NATIVE and key not in payload:
                payload[key] = value

        return json.dumps(payload, ensure_ascii=False)


logger, collector = fresh_logger("demo.json")
collector.setFormatter(JsonFormatter())

logger.info("платёж прошёл", extra={"user_id": 42, "level": "ПОДМЕНА"})
print(collector.format(collector.records[-1]))

`level` остался `INFO`, несмотря на `extra`. Служебных полей в выводе
нет. Ровно это и просят в задаче.

In [ ]:
logger, collector = fresh_logger("demo.json.exc")
collector.setFormatter(JsonFormatter())

try:
    1 / 0
except ZeroDivisionError:
    logger.exception("не смог поделить")

line = collector.format(collector.records[-1])
parsed = json.loads(line)
print("ключи:", sorted(parsed))
print("\ntraceback (первая строка):", parsed["traceback"].splitlines()[0])
print("\nвся запись — одна строка:", "\n" not in line)

## 6. `exception` против `error(exc)`

Разница в том, что попадёт в запись.

In [ ]:
logger, collector = fresh_logger("demo.exc")

try:
    1 / 0
except ZeroDivisionError as exc:
    logger.error(exc)

try:
    1 / 0
except ZeroDivisionError:
    logger.exception("не смог поделить")

for record in collector.records:
    print(f"{record.getMessage():22} exc_info: {'есть' if record.exc_info else 'НЕТ'}")

`logger.error(exc)` кладёт в запись только текст исключения. Ни файла,
ни строки, ни стека — `exc_info` пуст, и форматтер не выведет traceback.

**Внутри `except` — всегда `logger.exception()`.** Он сам подхватывает
текущее исключение.

## 7. Перехват записей в тестах

Логи — часть поведения, и их проверяют. В pytest для этого есть фикстура
`caplog`, но полезно понимать, что под ней: обычный хендлер, подсунутый
логгеру. Ровно такой, как `Collector` из первой ячейки.

В домашней задаче `expect_log` этот хендлер вешает autouse-фикстура, а
проверяет результат хук — связка из семинара.

```python
def test_warns(caplog):
    with caplog.at_level(logging.WARNING, logger="minipipe.prep"):
        minmax_scale(np.array([3.0, 3.0, 3.0]))

    assert "постоянна" in caplog.text
    assert caplog.records[-1].levelno == logging.WARNING
```

`caplog.text` — склеенный текст, `caplog.records` — сами объекты
`LogRecord`. Без `at_level` запись может не дойти: у логгера свой уровень.

## 8. `contextvars` — контекст без передачи

Сквозной `request_id` во всех записях — обычное требование. Передавать
его параметром через десять функций не хочется.

`ContextVar` выглядит как глобальная переменная, но глобальной не
является: у каждой «одновременной работы» своё значение. Обычную
глобальную такие работы затирают друг другу, а тут кто положил — тот и
прочитает.

In [ ]:
import asyncio
from contextvars import ContextVar

request_id: ContextVar[str] = ContextVar("request_id", default="-")


class ContextFilter(logging.Filter):
    """Дописывает текущий request_id в каждую запись."""

    def filter(self, record: logging.LogRecord) -> bool:
        record.request_id = request_id.get()
        return True


logger, collector = fresh_logger("demo.ctx")
logger.addFilter(ContextFilter())


def load_profile() -> None:
    logger.info("profile_loaded")     # про request_id ничего не знает


async def handle(rid: str) -> None:
    request_id.set(rid)
    logger.info("start")
    await asyncio.sleep(0)            # отдаём управление другому запросу
    load_profile()


async def main() -> None:
    await asyncio.gather(handle("req-1"), handle("req-2"))


await main()

for record in collector.records:
    print(f"  {record.request_id}  {record.getMessage()}")

Два запроса обрабатывались вперемешку, и `request_id` у каждого свой —
хотя `load_profile` о нём не знает и ничего не принимает.

Как устроена сама «одновременность» — потоки, корутины — разберём в
блоке про concurrency.